<a href="https://colab.research.google.com/github/German-rl/Actas_Cualitativas/blob/main/Actas_Cualitativas_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# INSTALACIÓN DE DEPENDENCIAS
# ==========================================

!apt-get update -qq
!apt-get install libreoffice -y -qq

!pip install openpyxl pandas
# ==========================================
# LIBRERÍAS
# ==========================================

import os
import zipfile
import subprocess

import pandas as pd

from openpyxl import load_workbook

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.11_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.11) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.11_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.11) ...
Selecting previously unselected package libuno-sal3.
Preparing to unpack .../002-libuno-sal3_1%3a7.3.7-0ubuntu0.22.04.11_amd64.deb ...
Unpacking libuno-sal3 (1:7.3.7-0ubuntu0.22.04.11) ...
Selecting previously unsele

In [2]:
# ==========================================
# CONFIGURACIÓN DE LA PLANTILLA
# ==========================================

CONFIG = {

    # Datos del alumno
    "nombre": "C4",
    "matricula": "C5",
    "grupo": "G3",

    # Datos generales
    "fecha": "C2",
    "materia": "C3",
    "semestre": "F2",

    # Resultado final
    "calificacion_numerica": "C21",
    "calificacion_cualitativa": "F21",

    # Comentario docente
    "comentario": "A12",
    "firma": "A24",

    # Recomendación
    "recomendaciones": "B15",

    # Indicadores
    "indicadores": {

        "indicador_1": {
            "texto": "A8",
            "optimo": "E8",
            "suficiente": "F8",
            "insuficiente": "G8"
        },

        "indicador_2": {
            "texto": "A9",
            "optimo": "E9",
            "suficiente": "F9",
            "insuficiente": "G9"
        },

        "indicador_3": {
            "texto": "A10",
            "optimo": "E10",
            "suficiente": "F10",
            "insuficiente": "G10"
        }
    }
}

In [3]:
# ==========================================
# INDICADORES
# ==========================================
INDICADORES = {

    "Química inorgánica": [

        "Analiza los conceptos generales de la química para interpretar el comportamiento de la materia y su estructura, así como los mecanismos mediante los cuales la materia se transforma, y establece la relación que existe entre la estructura y las propiedades de la materia.",

        "Utiliza los conocimientos de la química para construir las estructuras de sustancias tanto orgánicas como inorgánicas, así como su comportamiento químico y físico, para establecer relaciones entre los comportamientos y funciones biológicas de los seres vivos.",

        "Ø"
    ],

    "Química orgánica": [

        "Analiza los conceptos generales de la química del carbono , le permitan abordar y comprender la naturaleza de las moléculas así como de las estructuras que se encuentran en las células vivas y su entendimiento de la función biológica de dichas moléculas y estructuras",

        "Revisa, integra y hace uso de los conocimientos de la química orgánica para construir las estructuras orgánicas como inorgánicas, así como su comportamiento químico y físico, es capaz de establecer relaciones entre los comportamientos y funciones biológicas de los seres vivos. ",

        "Ø"
    ]
}

In [4]:
# ==========================================
# FUNCION MARCAR INDICADORES
# ==========================================
def marcar_indicadores(ws, calificacion):

    calificacion = round(calificacion)

    if calificacion >= 9:
        nivel = "optimo"

    elif calificacion >= 7:
        nivel = "suficiente"

    else:
        nivel = "insuficiente"

    for indicador in CONFIG["indicadores"].values():

        celda = indicador[nivel]

        ws[celda] = "X"

In [5]:
# ==========================================
# LIMPIAR INDICADORES
# ==========================================
def limpiar_indicadores(ws):

    for indicador in CONFIG["indicadores"].values():

        ws[indicador["optimo"]] = ""

        ws[indicador["suficiente"]] = ""

        ws[indicador["insuficiente"]] = ""

In [6]:
# ==========================================
# FUNCION ESCRIBIR INDICADORES
# ==========================================
def escribir_indicadores(ws, materia):

    lista = INDICADORES[materia]

    for i, texto in enumerate(lista, start=1):

        celda = CONFIG["indicadores"][f"indicador_{i}"]["texto"]

        ws[celda] = texto

In [7]:
# ==========================================
# FUNCION COMENTARIOS
# ==========================================
def obtener_comentario(calificacion):
    """
    Devuelve el comentario
    correspondiente.
    """

    if calificacion >= 9:
        return "Excelente trabajo, sigue así."

    elif calificacion >= 7:
        return "Tú puedes mejorar, confía en ti y tu potencial."

    else:
        return "Tienes que esforzarte más en la siguiente oportunidad."

In [8]:
# ==========================================
# FUNCION CERTIFICACION
# ==========================================
def obtener_certificacion(calificacion):
    """
    Determina si el alumno
    obtiene certificación.
    """

    if calificacion >= 7:
        return "CERTIFICADO"

    return "NO CERTIFICADO"

In [9]:
# ==========================================
# FUNCION RECOMENDACIONES
# ==========================================
def obtener_recomendaciones(calificacion):
    """
    Devuelve el comentario
    correspondiente.
    """

    if calificacion >= 9:
        return "Tu desempeño fue sobresaliente"

    elif calificacion >= 7:
        return "En tal virtud, se considera que Sí cumples con los indicadores de evaluación para certificar el curso"

    else:
        return "Cursa nuevamente la materia para completar los propósitos indispensables y después volve a presentarse a certificación"

In [10]:
# ==========================================
# FUNCION solicitar datos
# ==========================================
def solicitar_datos_generales():

    print("\nMaterias disponibles:\n")

    for materia in INDICADORES.keys():
        print("-", materia)

    materia = input(
        "\nSeleccione la materia exactamente como aparece: "
    )

    datos = {

        "fecha": input("Fecha: "),
        "materia": materia,
        "semestre": input("Semestre: "),
        "grupo": input("Grupo: "),
        "profesor": input("Profesor: ")
    }

    return datos

In [11]:
# ==========================================
# funcion descargar plantilla
# ==========================================
import requests

PLANTILLA = "Plantilla.xlsx"
URL_PLANTILLA = "https://raw.githubusercontent.com/German-rl/Actas_Cualitativas/main/Plantilla.xlsx"

def descargar_plantilla():

    respuesta = requests.get(URL_PLANTILLA)

    respuesta.raise_for_status()  # Verifica que la descarga fue exitosa

    with open(PLANTILLA, "wb") as f:
        f.write(respuesta.content)

    print("✅ Plantilla descargada correctamente.")

In [12]:
# ==========================================
# FUNCION LLENAR ACTA
# ==========================================
def llenar_acta(
        plantilla,
        nombre,
        matricula,
        calificacion,
        datos_generales,
        archivo_salida):

    wb = load_workbook(plantilla)

    ws = wb["Alumno"]

    # ------------------------
    # Datos generales
    # ------------------------

    ws[CONFIG["fecha"]] = datos_generales["fecha"]

    ws[CONFIG["materia"]] = datos_generales["materia"]

    ws[CONFIG["semestre"]] = datos_generales["semestre"]

    ws[CONFIG["grupo"]] = datos_generales["grupo"]

    ws[CONFIG["firma"]] = datos_generales["profesor"]

    # ------------------------
    # Datos alumno
    # ------------------------

    ws[CONFIG["nombre"]] = nombre

    ws[CONFIG["matricula"]] = matricula

    # ------------------------
    # Calificación
    # ------------------------

    calificacion = round(calificacion)

    ws[CONFIG["calificacion_numerica"]] = (
        calificacion
    )

    ws[CONFIG["calificacion_cualitativa"]] = (
        obtener_certificacion(calificacion)
    )

    # ------------------------
    # Comentarios
    # ------------------------

    ws[CONFIG["comentario"]] = (
        obtener_comentario(calificacion)
    )
    ws[CONFIG["recomendaciones"]] = (
        obtener_recomendaciones(calificacion)
    )

    # ------------------------
    # Indicadores
    # ------------------------

    limpiar_indicadores(ws)


    escribir_indicadores(
        ws,
        datos_generales["materia"]
    )

    marcar_indicadores(
        ws,
        calificacion
    )

    wb.save(archivo_salida)

In [13]:
# ==========================================
# FUNCION EXCEL A PDF
# ==========================================
def excel_a_pdf(
        archivo_excel,
        carpeta_salida):

    subprocess.run([
        "libreoffice",
        "--headless",
        "--convert-to",
        "pdf",
        archivo_excel,
        "--outdir",
        carpeta_salida
    ])

In [14]:
# ==========================================
# Subir documento
# ==========================================
from google.colab import files
import os

def subir_calificaciones():

    print(
        "Seleccione el archivo de calificaciones"
    )

    uploaded = files.upload()

    nombre_original = next(
        iter(uploaded)
    )

    if nombre_original != (
        "Calificaciones.xlsx"
    ):
        os.rename(
            nombre_original,
            "Calificaciones.xlsx"
        )

    return "Calificaciones.xlsx"

In [15]:
# ==========================================
# CREAR ACTA
# ==========================================
# ==========================================
# CREAR ACTAS
# ==========================================
def generar_actas():

    # ------------------------
    # Descargar plantilla
    # ------------------------
    descargar_plantilla()

    plantilla_excel = PLANTILLA

    # ------------------------
    # Solicitar archivo de calificaciones
    # ------------------------
    archivo_calificaciones = subir_calificaciones()

    # ------------------------
    # Datos generales
    # ------------------------
    datos_generales = solicitar_datos_generales()

    # ------------------------
    # Crear carpeta de salida
    # ------------------------
    os.makedirs(
        "PDFs",
        exist_ok=True
    )

    # ------------------------
    # Leer calificaciones
    # ------------------------
    df = pd.read_excel(
        archivo_calificaciones
    )

    # ------------------------
    # Generar una acta por alumno
    # ------------------------
    for _, alumno in df.iterrows():

        nombre = alumno["Nombre"]

        matricula = alumno["Matricula"]

        calificacion = alumno["Calificación"]

        nombre_archivo = (
            nombre
            .replace(" ", "_")
            .replace("/", "_")
        )

        archivo_excel_salida = (
            f"PDFs/{nombre_archivo}.xlsx"
        )

        llenar_acta(
            plantilla_excel,
            nombre,
            matricula,
            calificacion,
            datos_generales,
            archivo_excel_salida
        )

        excel_a_pdf(
            archivo_excel_salida,
            "PDFs"
        )

        print(f"Generado: {nombre}")

In [16]:
# ==========================================
# FUNCION CREAR ZIP
# ==========================================
def crear_zip():

    with zipfile.ZipFile(
        "ActasPDF.zip",
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        for archivo in os.listdir("PDFs"):

            if archivo.endswith(".pdf"):

                z.write(
                    os.path.join(
                        "PDFs",
                        archivo
                    ),
                    archivo
                )

    print("ZIP generado.")

In [18]:

generar_actas()

crear_zip()

✅ Plantilla descargada correctamente.
Seleccione el archivo de calificaciones


Saving archivo_calificaciones.xlsx to archivo_calificaciones.xlsx

Materias disponibles:

- Química inorgánica
- Química orgánica

Seleccione la materia exactamente como aparece: Química inorgánica
Fecha: 01/07/2026
Semestre: 26-I
Grupo: 1103
Profesor: Germán RL


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Generado: Castro Fiesco America Lucero


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


...

Generado: Gómez Hernández Mónica Fernanda


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Generado: Layseca Vargas Angel


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Generado: Rodriguez Oliva Litzy Jennifer
ZIP generado.
